# Washington DC Office and Multifamily Stock Analysis

This notebook searches ComStock office buildings and ResStock multifamily dwelling units in Washington, DC, compares stock characteristics, downloads a few of the most energy-intensive records, plots demand and monthly end-use profiles, and generates measure recommendations from the observed energy drivers.

In [ ]:
import os
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from comstock_processor import ComStockProcessor
from resstock_processor import ResStockProcessor

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.mode.chained_assignment = None

STATE = "DC"
UPGRADE = "0"
EUI_COL = "out.site_energy.total.energy_consumption_intensity..kwh_per_ft2"
SQFT_COL = "in.sqft..ft2"
TOTAL_SITE_COL = "out.site_energy.total.energy_consumption..kwh"

dataset_dir = Path(os.environ.get("BUILDSTOCK_DATA_DIR", Path.cwd() / "datasets"))
comstock_dir = dataset_dir / "comstock" / "dc_stock_analysis"
resstock_dir = dataset_dir / "resstock" / "dc_stock_analysis"
base_dir = dataset_dir / "dc_stock_analysis"
comstock_ts_dir = comstock_dir / "time_series"
resstock_ts_dir = resstock_dir / "time_series"
figures_dir = Path.cwd() / "figures" / "dc_stock_analysis"

for path in [comstock_dir, resstock_dir, comstock_ts_dir, resstock_ts_dir, figures_dir]:
    path.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
print(f"Working directory: {base_dir}")

## Search Washington, DC Stock

ComStock represents whole commercial buildings, so the office search combines `SmallOffice`, `MediumOffice`, and `LargeOffice`. ResStock represents individual residential dwelling units, not whole multifamily buildings, so the multifamily query combines the two multifamily housing categories. A ResStock record includes its unit's floor area, location within the building, and the building's unit-count category; it does not include a shared physical-building identifier that can reliably aggregate sampled units into a whole building.

In [ ]:
def load_comstock_offices() -> pd.DataFrame:
    frames = []
    for building_type in ["SmallOffice", "MediumOffice", "LargeOffice"]:
        processor = ComStockProcessor(
            state=STATE,
            county_name="All",
            building_type=building_type,
            upgrade=UPGRADE,
            base_dir=comstock_dir,
        )
        frame = processor.process_metadata(save_dir=comstock_dir).copy()
        frame["stock"] = "ComStock"
        frame["search_group"] = "Office"
        frame["building_category"] = frame["in.comstock_building_type"]
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def load_resstock_multifamily() -> pd.DataFrame:
    frames = []
    for building_type in ["Multi-Family with 2 - 4 Units", "Multi-Family with 5+ Units"]:
        processor = ResStockProcessor(
            state=STATE,
            county_name="All",
            building_type=building_type,
            upgrade=UPGRADE,
            base_dir=resstock_dir,
        )
        frame = processor.process_metadata(save_dir=resstock_dir).copy()
        frame["stock"] = "ResStock"
        frame["search_group"] = "Multifamily"
        frame["building_category"] = frame["in.geometry_building_type_recs"]
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


office_df = load_comstock_offices()
multifamily_df = load_resstock_multifamily()
stock_df = pd.concat([office_df, multifamily_df], ignore_index=True)

stock_df[EUI_COL] = pd.to_numeric(stock_df[EUI_COL], errors="coerce")
stock_df[SQFT_COL] = pd.to_numeric(stock_df[SQFT_COL], errors="coerce")

summary_cols = ["stock", "search_group", "building_category", "bldg_id", "in.state", "in.county_name", SQFT_COL, EUI_COL]
display(stock_df[summary_cols].head())
print(f"Office records: {len(office_df):,}")
print(f"Multifamily dwelling-unit records: {len(multifamily_df):,}")

record_counts = stock_df.groupby(["search_group", "building_category"]).size().rename("modeled_records").reset_index()
display(record_counts)

unit_context_cols = [
    "bldg_id",
    "in.geometry_building_number_units_mf",
    "in.geometry_building_horizontal_location_mf",
    "in.geometry_building_level_mf",
    SQFT_COL,
    EUI_COL,
]
display(multifamily_df[unit_context_cols].head().rename(columns={"bldg_id": "dwelling_unit_id"}))

## Stock Characteristics

The first plots compare how many records are available by category, how floor area relates to site-energy intensity, and how the high-energy tail differs between office buildings and multifamily units.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

category_counts = stock_df.groupby(["search_group", "building_category"]).size().sort_values()
category_colors = ["#2a9d8f" if group == "Multifamily" else "#4c78a8" for group, _ in category_counts.index]
category_counts.plot(kind="barh", ax=axes[0], color=category_colors)
axes[0].set_title("Modeled Records by Category")
axes[0].set_xlabel("Buildings (office) or dwelling units (multifamily)")
axes[0].set_ylabel("")

plot_styles = {
    "Office": {"color": "#4c78a8", "marker": "o", "label": "Office buildings", "zorder": 1},
    "Multifamily": {"color": "#2a9d8f", "marker": "x", "label": "Multifamily dwelling units", "zorder": 2},
}
for search_group, frame in stock_df.groupby("search_group"):
    axes[1].scatter(frame[SQFT_COL], frame[EUI_COL], s=24, alpha=0.65, **plot_styles[search_group])
axes[1].set_xscale("log")
axes[1].set_title("Floor Area vs. Site EUI")
axes[1].set_xlabel("Floor area (ft2, log scale)")
axes[1].set_ylabel("Site EUI (kWh/ft2)")
axes[1].legend()

stock_df.boxplot(column=EUI_COL, by="search_group", ax=axes[2], grid=False)
axes[2].set_title("Site EUI by Record Type")
axes[2].set_xlabel("")
axes[2].set_ylabel("Site EUI (kWh/ft2)")
fig.suptitle("")
fig.tight_layout()
plt.show()

annual_meter_cols = [
    "out.electricity.total.energy_consumption..kwh",
    "out.natural_gas.total.energy_consumption..kwh",
    "out.district_cooling.total.energy_consumption..kwh",
    "out.district_heating.total.energy_consumption..kwh",
    "out.other_fuel.total.energy_consumption..kwh",
]
available_meter_cols = [col for col in annual_meter_cols if col in stock_df.columns]
annual_meters = stock_df.groupby("search_group")[available_meter_cols].median().T
annual_meters.index = annual_meters.index.str.replace("out.", "", regex=False).str.replace(
    ".total.energy_consumption..kwh", "", regex=False
)
annual_meters.plot(kind="bar", figsize=(10, 5), color=["#4c78a8", "#f58518"])
plt.title("Median Annual Consumption per Modeled Record")
plt.ylabel("Annual energy consumption (kWh per record)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Select and Download High-Intensity Records

The next cell selects the three highest-site-EUI records from each stock dataset. For ComStock these are whole office buildings; for ResStock these are individual multifamily dwelling units. The downloads stay intentionally small while still showing record-level time-series analysis.

In [ ]:
def top_intensive(frame: pd.DataFrame, stock_name: str, n: int = 3) -> pd.DataFrame:
    return frame.dropna(subset=[EUI_COL]).sort_values(EUI_COL, ascending=False).head(n).assign(stock=stock_name)


selected_offices = top_intensive(office_df, "ComStock")
selected_multifamily = top_intensive(multifamily_df, "ResStock")
selected_offices["bldg_id"] = selected_offices["bldg_id"].astype(str)
selected_multifamily["bldg_id"] = selected_multifamily["bldg_id"].astype(str)
selected = pd.concat([selected_offices, selected_multifamily], ignore_index=True)

selected_display_cols = ["stock", "search_group", "building_category", "bldg_id", SQFT_COL, EUI_COL, TOTAL_SITE_COL]
selected_display_cols = [col for col in selected_display_cols if col in selected.columns]
display(selected[selected_display_cols])

In [ ]:
comstock_download_processor = ComStockProcessor(state=STATE, county_name="All", building_type="All", upgrade=UPGRADE, base_dir=comstock_dir)
resstock_download_processor = ResStockProcessor(state=STATE, county_name="All", building_type="All", upgrade=UPGRADE, base_dir=resstock_dir)

comstock_paths, comstock_ids = comstock_download_processor.process_building_time_series(selected_offices, save_dir=comstock_ts_dir)
resstock_paths, resstock_ids = resstock_download_processor.process_building_time_series(selected_multifamily, save_dir=resstock_ts_dir)

downloads = pd.DataFrame(
    {
        "stock": ["ComStock"] * len(comstock_paths) + ["ResStock"] * len(resstock_paths),
        "bldg_id": comstock_ids + resstock_ids,
        "path": [str(path) for path in comstock_paths + resstock_paths],
    }
)
display(downloads)

## Demand Profiles and Monthly End Uses

The time-series files contain interval energy consumption columns. The demand plot treats interval electricity consumption as a demand proxy; when the source interval is hourly, the units read directly as average kW for that hour.

In [ ]:
def read_time_series(paths: list[Path], building_ids: list[str], stock_name: str) -> pd.DataFrame:
    frames = []
    for path, building_id in zip(paths, building_ids):
        frame = pd.read_parquet(path)
        frame["timestamp"] = pd.to_datetime(frame["timestamp"])
        frame["bldg_id"] = str(building_id)
        frame["stock"] = stock_name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


ts_df = pd.concat(
    [
        read_time_series(comstock_paths, comstock_ids, "ComStock"),
        read_time_series(resstock_paths, resstock_ids, "ResStock"),
    ],
    ignore_index=True,
)

energy_cols = [col for col in ts_df.columns if col.startswith("out.") and col.endswith(".energy_consumption")]
print(f"Loaded {len(ts_df):,} time-series rows and {len(energy_cols)} energy columns.")
display(ts_df[["stock", "bldg_id", "timestamp", *energy_cols[:5]]].head())

In [ ]:
electricity_total_col = "out.electricity.total.energy_consumption"
if electricity_total_col not in ts_df.columns:
    total_cols = [col for col in energy_cols if ".total." in col]
    raise KeyError(f"Expected {electricity_total_col} in time-series data. Available total columns: {total_cols}")

daily_profile = (
    ts_df.assign(hour=ts_df["timestamp"].dt.hour).groupby(["stock", "bldg_id", "hour"], as_index=False)[electricity_total_col].mean()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, stock_name in zip(axes, ["ComStock", "ResStock"]):
    subset = daily_profile[daily_profile["stock"] == stock_name]
    for building_id, frame in subset.groupby("bldg_id"):
        ax.plot(frame["hour"], frame[electricity_total_col], marker="o", linewidth=1.5, label=building_id)
    ax.set_title(f"{stock_name} High-EUI Records")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Mean electricity use per interval (kWh)")
    ax.legend(title="bldg_id", fontsize=8)
fig.suptitle("Average Daily Electricity Demand Profiles")
fig.tight_layout()
plt.show()

In [ ]:
def parse_end_use_column(column: str) -> tuple[str, str] | None:
    match = re.fullmatch(r"out\.([^.]+)\.(.+)\.energy_consumption", column)
    if match is None:
        return None
    fuel, end_use = match.groups()
    if end_use == "total":
        return None
    return fuel, end_use


parsed_cols = {col: parse_end_use_column(col) for col in energy_cols}
end_use_cols = [col for col, parsed in parsed_cols.items() if parsed is not None]

monthly_long = ts_df[["stock", "bldg_id", "timestamp", *end_use_cols]].melt(
    id_vars=["stock", "bldg_id", "timestamp"],
    var_name="column",
    value_name="energy_kwh",
)
monthly_long["month"] = monthly_long["timestamp"].dt.month
monthly_long["fuel"] = monthly_long["column"].map(lambda col: parsed_cols[col][0])
monthly_long["end_use"] = monthly_long["column"].map(lambda col: parsed_cols[col][1])
monthly_long["fuel_end_use"] = monthly_long["fuel"] + " - " + monthly_long["end_use"]

monthly_end_use = monthly_long.groupby(["stock", "month", "fuel_end_use"], as_index=False)["energy_kwh"].sum()

for stock_name, frame in monthly_end_use.groupby("stock"):
    top_end_uses = frame.groupby("fuel_end_use")["energy_kwh"].sum().nlargest(8).index
    pivot = (
        frame[frame["fuel_end_use"].isin(top_end_uses)]
        .pivot_table(index="month", columns="fuel_end_use", values="energy_kwh", aggfunc="sum")
        .fillna(0)
        .sort_index()
    )
    pivot.plot(kind="area", stacked=True, figsize=(11, 5), linewidth=0)
    plt.title(f"{stock_name}: Monthly End-Use Consumption for Selected High-EUI Records")
    plt.xlabel("Month")
    plt.ylabel("Energy consumption (kWh)")
    plt.legend(title="Fuel / end use", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## Recommend Measures

The recommendation logic below is deliberately transparent: it identifies each selected record's largest time-series end use and fuel balance, then maps those drivers to measure-package keywords available in the relevant stock dataset's upgrade lookup.

In [ ]:
comstock_upgrades = comstock_download_processor.list_upgrades(comstock_dir)
resstock_upgrades = resstock_download_processor.list_upgrades(resstock_dir)


def matching_upgrades(upgrades: dict[str, str], keywords: list[str], limit: int = 4) -> str:
    matches = []
    for upgrade_id, name in upgrades.items():
        lowered = name.lower()
        if any(keyword.lower() in lowered for keyword in keywords):
            matches.append(f"{upgrade_id}: {name}")
    return "; ".join(matches[:limit]) if matches else "No direct package-name match found"


def recommendation_focus(top_end_use: str, electricity_share: float) -> tuple[str, list[str]]:
    lowered = top_end_use.lower()
    if "cool" in lowered:
        return "Cooling load reduction and high-efficiency HVAC", ["cool", "rtu", "heat pump", "economizer"]
    if "heat" in lowered or electricity_share < 0.45:
        return "Heating fuel reduction, electrification, and envelope improvements", [
            "heat pump",
            "envelope",
            "insulation",
            "air seal",
        ]
    if "light" in lowered:
        return "Lighting power and controls", ["lighting", "occupancy", "control"]
    if "water" in lowered:
        return "Domestic hot water efficiency", ["water heater", "heat pump water", "dhw"]
    if electricity_share >= 0.70:
        return "Electric peak and plug/process load management", ["controls", "plug", "equipment", "demand"]
    return "Whole-building efficiency package review", ["package", "efficiency", "upgrade"]


annual_selected = selected.set_index(["stock", "bldg_id"])
end_use_by_building = (
    monthly_long.groupby(["stock", "bldg_id", "fuel_end_use"], as_index=False)["energy_kwh"]
    .sum()
    .sort_values(["stock", "bldg_id", "energy_kwh"], ascending=[True, True, False])
)

recommendations = []
for (stock_name, building_id), group in end_use_by_building.groupby(["stock", "bldg_id"]):
    top_end_use = group.iloc[0]["fuel_end_use"]
    row = annual_selected.loc[(stock_name, building_id)]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    electricity = pd.to_numeric(row.get("out.electricity.total.energy_consumption..kwh", 0), errors="coerce")
    natural_gas = pd.to_numeric(row.get("out.natural_gas.total.energy_consumption..kwh", 0), errors="coerce")
    electricity = 0 if pd.isna(electricity) else float(electricity)
    natural_gas = 0 if pd.isna(natural_gas) else float(natural_gas)
    electricity_share = electricity / max(electricity + natural_gas, 1)
    focus, keywords = recommendation_focus(top_end_use, electricity_share)
    upgrades = comstock_upgrades if stock_name == "ComStock" else resstock_upgrades
    recommendations.append(
        {
            "stock": stock_name,
            "bldg_id": building_id,
            "category": row["building_category"],
            "site_eui_kwh_per_ft2": row[EUI_COL],
            "dominant_time_series_end_use": top_end_use,
            "electricity_share_of_elec_plus_gas": round(electricity_share, 2),
            "recommended_measure_focus": focus,
            "matching_available_upgrade_packages": matching_upgrades(upgrades, keywords),
        }
    )

recommendations_df = pd.DataFrame(recommendations).sort_values(["stock", "site_eui_kwh_per_ft2"], ascending=[True, False])
display(recommendations_df)

## Notes for Adapting

- Use `county_name="All"` for DC because the district is a single state-level search target in these releases.
- Increase the `head(n)` value in the high-EUI selection cell to download more time-series files.
- For a retrofit study, use `process_metadata_for_upgrades()` to compare the same records under specific upgrade packages before and after applying a measure.